<a href="https://colab.research.google.com/github/luca-fce/siniestros-futbol-caba/blob/main/notebook/3_3_transformaciones_y_resumen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.3 Transformaciones y resumen estadístico

Este notebook corrige los problemas detectados en **3.1**, arma la medición que pide la pregunta de **3.2** y la resume con estadísticos agrupados y pruebas de significación. Los archivos que genera se usan en **3.4** para los gráficos.

**Orden:**
1. Limpieza de siniestros
2. Preparación de los partidos de River y Boca
3. Preparación de los partidos de la Selección
4. Cruce: siniestros en las 12 h posteriores a cada partido vs. lo esperado
5. Resumen agrupado (`groupby`)
6. Pruebas de significación
7. Robustez
8. Exportación

> **Uso de IA.** Todas las celdas de este notebook se elaboraron con asistencia de Claude (Anthropic). Prompt utilizado: «Limpiá las bases de siniestros, de River y Boca y de la Selección (hora argentina, penales, horarios reales), y cruzalas contando los siniestros en las 12 h posteriores a cada partido contra un control de misma hora, día de la semana y año; resumí con groupby, testeá si las derrotas aumentan los siniestros y probá la robustez del resultado». El grupo revisó el código, verificó los resultados contra las fuentes y validó las interpretaciones.

Librerías y rutas.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm

REPO = 'https://raw.githubusercontent.com/luca-fce/siniestros-futbol-caba/main/'
RUTA_SIN = REPO + 'datos/Siniestros%20Viales%202019%20-%202025/'
RUTA_RB = REPO + 'datos/Temporadas%20River%20-%20Boca/'
RUTA_INT = REPO + 'datos/Resultados%20Internacionales%201872%20-%202026/'
ORDEN = ['Derrota', 'Empate', 'Victoria']

## 1. Limpieza de siniestros

**Transformaciones y por qué:**

1. **Unificar faltantes:** `SD`, `#¡REF!` y vacíos pasan a `NaN`, para que se traten como datos faltantes y no como una categoría.
2. **Convertir tipos:** fechas a `datetime`, cantidades a enteros y coordenadas a decimales.
3. **Recalcular el total de víctimas** donde falta, como leves + graves + mortales (en los datos completos esa suma coincide con el total en el 99,99 % de los casos).
4. **Crear `fecha_hora`** uniendo fecha y hora: es la variable que permite ubicar cada siniestro en la ventana de 12 h.
5. **Normalizar textos y comunas** (mayúsculas; comuna como número del 1 al 15).
6. **Filtrar 2021–2025**, el período de los partidos. Además, 2020 estuvo distorsionado por la cuarentena.

Los duplicados de víctimas no se eliminan (ver 3.1).

In [2]:
hechos_raw = pd.read_csv(RUTA_SIN + 'siniestros_viales_hechos.csv', sep=';', encoding='utf-8-sig', dtype=str)
victimas_raw = pd.read_csv(RUTA_SIN + 'siniestros_viales_victimas.csv', sep=';', encoding='utf-8-sig', dtype=str)

h = hechos_raw.copy()
# 1. Faltantes
h = h.replace({'SD': np.nan, '#¡REF!': np.nan, '': np.nan})
h['geocodificacion_plana'] = h['geocodificacion_plana'].replace('POINT (SD SD)', np.nan)
# 2. Tipos
h['fecha_siniestro'] = pd.to_datetime(h['fecha_siniestro'], format='%Y-%m-%d')  # acepta '2025-1-9'
for c in ['numero_total_de_victimas', 'numero_victimas_leve_siniestro', 'numero_victimas_grave_siniestro',
          'numero_victimas_mortal_siniestro', 'anio_siniestro', 'mes_siniestro', 'dia_siniestro', 'rango_horario']:
    h[c] = pd.to_numeric(h[c], errors='coerce').astype('Int64')
for c in ['latitud_siniestro', 'longitud_siniestro']:
    h[c] = pd.to_numeric(h[c], errors='coerce')
# 3. Total de víctimas faltante
suma = h['numero_victimas_leve_siniestro'] + h['numero_victimas_grave_siniestro'] + h['numero_victimas_mortal_siniestro']
h['numero_total_de_victimas'] = h['numero_total_de_victimas'].fillna(suma)
# año, mes y día derivados de la fecha (había 1 fila inconsistente)
h['anio_siniestro'] = h['fecha_siniestro'].dt.year
h['mes_siniestro'] = h['fecha_siniestro'].dt.month
h['dia_siniestro'] = h['fecha_siniestro'].dt.day
# 4. Fecha y hora
h['fecha_hora'] = pd.to_datetime(h['fecha_siniestro'].dt.strftime('%Y-%m-%d') + ' ' + h['hora_siniestro'],
                                 format='%Y-%m-%d %H:%M:%S', errors='coerce')
h['tiene_hora'] = h['fecha_hora'].notna()
h['dia_semana'] = h['fecha_siniestro'].dt.dayofweek  # 0 = lunes ... 6 = domingo
# 5. Textos y comunas
for c in ['tipo_de_via_siniestro', 'participantes_siniestro', 'modo_desplazamiento_victima',
          'contraparte_siniestro', 'gravedad_siniestro', 'direccion_normalizada_siniestro']:
    h[c] = h[c].str.strip().str.upper()
h['comuna_siniestro'] = pd.to_numeric(h['comuna_siniestro'].str.extract(r'(\d+)')[0], errors='coerce').astype('Int64')
h = h.rename(columns={'comuna_siniestro': 'comuna'})
# 6. Filtro
hechos = h[h['anio_siniestro'].between(2021, 2025)].sort_values('fecha_hora').reset_index(drop=True)

print('Siniestros 2021-2025:', len(hechos), '| sin hora:', (~hechos['tiene_hora']).sum())
print('Contraparte después de normalizar:', sorted(hechos['contraparte_siniestro'].dropna().unique()))

Siniestros 2021-2025: 49602 | sin hora: 77
Contraparte después de normalizar: ['AUTO', 'BICICLETA', 'CAMION', 'CAMIONETA', 'MONOPATIN', 'MOTO', 'MOVIL', 'MULTIPLE', 'OBJETO FIJO', 'OTRO', 'PEATON', 'TAXI', 'TRANSPORTE PUBLICO', 'TREN', 'UTILITARIO']


Misma limpieza para la tabla de víctimas: se quitan las filas vacías, se corrige el nombre de la columna de gravedad, la edad pasa a número y se normalizan los textos.

In [3]:
v = victimas_raw.dropna(how='all').copy()
v = v.rename(columns={'GRAVEdad_victima': 'gravedad_victima'})
v = v.replace({'SD': np.nan, 'sd': np.nan, '': np.nan})
v['fecha_siniestro'] = pd.to_datetime(v['fecha_siniestro'], format='%Y-%m-%d')
v['fecha_fallecimiento_victima'] = pd.to_datetime(v['fecha_fallecimiento_victima'], format='%Y-%m-%d', errors='coerce')
v['anio_siniestro'] = v['fecha_siniestro'].dt.year
v['edad_victima'] = pd.to_numeric(v['edad_victima'], errors='coerce').astype('Int64')
for c in ['modo_desplazamiento_victima', 'sexo_victima', 'gravedad_victima', 'rol_victima']:
    v[c] = v[c].str.strip().str.upper()
victimas = v[v['anio_siniestro'].between(2021, 2025)].reset_index(drop=True)

print('Víctimas 2021-2025:', len(victimas))
print('Víctimas sin hecho asociado:', (~victimas['id_siniestro'].isin(hechos['id_siniestro'])).sum())
print(victimas['gravedad_victima'].value_counts())

Víctimas 2021-2025: 56371
Víctimas sin hecho asociado: 0
gravedad_victima
LEVE      53296
GRAVE      2558
MORTAL      517
Name: count, dtype: int64


Serie de siniestros **por hora** (incluyendo las horas con cero), que es la base del cruce con los partidos:

In [4]:
siniestros_por_hora = (hechos.dropna(subset=['fecha_hora']).set_index('fecha_hora').resample('h').size()
                       .rename('siniestros').reset_index())
siniestros_por_hora['hora'] = siniestros_por_hora['fecha_hora'].dt.hour
siniestros_por_hora['dia_semana'] = siniestros_por_hora['fecha_hora'].dt.dayofweek
siniestros_por_hora.describe().round(2)

,fecha_hora,siniestros,hora,dia_semana
count,43820,43820.00,43820.00,43820.0
mean,2023-07-02 21:30:00,1.13,11.50,3.0
min,2021-01-01 00:00:00,0.00,0.00,0.0
25%,2022-04-02 10:45:00,0.00,5.00,1.0
50%,2023-07-02 21:30:00,1.00,11.00,3.0
75%,2024-10-01 08:15:00,2.00,17.00,5.0
max,2025-12-31 19:00:00,11.00,23.00,6.0
std,NaN,1.31,6.92,2.0


## 2. Partidos de River y Boca

**Transformaciones y por qué:**

1. **Hora argentina:** si el horario tiene paréntesis se usa ese valor; si no, la hora local ya es la argentina.
2. **Goles y penales separados** (`1 (4)` → 1 gol y 4 penales) y convertidos a números.
3. **Resultado final:** para el hincha, quedar eliminado por penales es una derrota. Los partidos definidos por penales se reclasifican según quién ganó la tanda.
4. **Inicio y fin:** fin = inicio + 1 h 55 min (90' + entretiempo + adicionado), +30 min si hubo alargue y +15 min si hubo penales.
5. **Rival sin código de país** y marca de **superclásico**.

In [5]:
archivos = {'Boca Juniors': 'Boca%20Juniors/boca_{}.csv', 'River Plate': 'River%20Plate/river_{}.csv'}
lista = []
for equipo, patron in archivos.items():
    for anio in range(2021, 2026):
        df = pd.read_csv(RUTA_RB + patron.format(anio))
        df.insert(0, 'equipo', equipo)
        df.insert(1, 'temporada', anio)
        lista.append(df)
p = pd.concat(lista, ignore_index=True)

# 1. Hora argentina
p['hora_arg'] = p['Time'].str.extract(r'\((\d{2}:\d{2})\)')[0].fillna(p['Time'].str[:5])
# 2. Goles y penales
for col, nombre in [('GF', 'a_favor'), ('GA', 'en_contra')]:
    partes = p[col].astype(str).str.extract(r'^\s*(\d+)\s*(?:\((\d+)\))?')
    p[f'goles_{nombre}'] = pd.to_numeric(partes[0]).astype('Int64')
    p[f'penales_{nombre}'] = pd.to_numeric(partes[1]).astype('Int64')
p['penales'] = p['penales_a_favor'].notna()
p['alargue'] = p['Notes'].fillna('').str.contains('extra time', case=False)
# 3. Resultado final
p['resultado'] = p['Result'].map({'W': 'Victoria', 'D': 'Empate', 'L': 'Derrota'})
p.loc[p['penales'] & (p['penales_a_favor'] > p['penales_en_contra']), 'resultado'] = 'Victoria'
p.loc[p['penales'] & (p['penales_a_favor'] < p['penales_en_contra']), 'resultado'] = 'Derrota'
# 4. Inicio y fin
p['inicio'] = pd.to_datetime(p['Date'] + ' ' + p['hora_arg'])
p['fin'] = (p['inicio'] + pd.Timedelta(minutes=115)
            + pd.to_timedelta(np.where(p['alargue'], 30, 0), unit='min')
            + pd.to_timedelta(np.where(p['penales'], 15, 0), unit='min'))
# 5. Rival y superclásico
p['rival'] = p['Opponent'].str.replace(r'^[a-z]{2} ', '', regex=True)
p['superclasico'] = p['rival'].isin(['River Plate', 'Boca Juniors'])

river_boca = (p.rename(columns={'Comp': 'competencia', 'Round': 'instancia', 'Venue': 'condicion'})
              [['equipo', 'temporada', 'inicio', 'fin', 'hora_arg', 'competencia', 'instancia', 'condicion', 'rival',
                'goles_a_favor', 'goles_en_contra', 'penales', 'penales_a_favor', 'penales_en_contra',
                'alargue', 'resultado', 'superclasico']]
              .sort_values('inicio').reset_index(drop=True))
print('Partidos:', len(river_boca), '| por penales:', river_boca['penales'].sum(),
      '| reclasificados de empate:', ((p['Result'] == 'D') & p['penales']).sum())
river_boca[river_boca['penales']][['equipo', 'inicio', 'rival', 'goles_a_favor', 'goles_en_contra',
                                   'penales_a_favor', 'penales_en_contra', 'resultado']].head(8)

Partidos: 501 | por penales: 17 | reclasificados de empate: 14


,equipo,inicio,rival,goles_a_favor,goles_en_contra,penales_a_favor,penales_en_contra,resultado
34,Boca Juniors,2021-05-16 17:30:00,River Plate,1,1,4,2,Victoria
35,River Plate,2021-05-16 17:30:00,Boca Juniors,1,1,2,4,Derrota
40,Boca Juniors,2021-05-31 15:00:00,Racing Club,0,0,2,4,Derrota
45,Boca Juniors,2021-07-20 19:15:00,Atlético Mineiro,0,0,1,3,Derrota
135,Boca Juniors,2022-05-14 17:00:00,Racing Club,0,0,6,5,Victoria
155,Boca Juniors,2022-07-05 21:30:00,Corinthians,0,0,5,6,Derrota
267,River Plate,2023-08-08 21:00:00,Internacional,1,2,8,9,Derrota
268,Boca Juniors,2023-08-09 21:00:00,Nacional,2,2,4,2,Victoria


## 3. Partidos de la Selección

**Transformaciones y por qué:**

1. **Filtro** de los partidos de Argentina entre 2021 y 2025 (oficiales y amistosos).
2. **Hora de inicio:** se toma de [Wikipedia — Argentina national football team results (2020–present)](https://en.wikipedia.org/wiki/Argentina_national_football_team_results_(2020%E2%80%93present)), que informa la hora local del estadio con su huso horario, y se pasa a **hora argentina (UTC−3**; Argentina no tiene horario de verano): `hora_arg = hora_local − (huso + 3)`. Ejemplo: Arabia Saudita en Qatar fue a las 13:00 (UTC+3) → **07:00** en Argentina. Para Argentina–Chile del 5/9/2024 en el Monumental la fuente indica UTC−4 por error; se usó 21:00, hora argentina.
3. **Goles desde el punto de vista de Argentina** y condición (local, visitante o neutral).
4. **Penales:** `shootouts.csv` indica quién ganó la tanda; ganar por penales cuenta como victoria y perder, como derrota.

In [6]:
internacionales = pd.read_csv(RUTA_INT + 'results.csv', parse_dates=['date'])
penales_raw = pd.read_csv(RUTA_INT + 'shootouts.csv', parse_dates=['date'])
# 1. Filtro
es_arg = (internacionales['home_team'] == 'Argentina') | (internacionales['away_team'] == 'Argentina')
arg = internacionales[es_arg & internacionales['date'].between('2021-01-01', '2025-12-31')].copy().reset_index(drop=True)

# 2. Horarios: fecha -> (hora local de inicio, huso horario UTC del estadio)
horarios = {
    '2021-06-03': ('21:00', -3), '2021-06-08': ('18:00', -5), '2021-06-14': ('18:00', -3),
    '2021-06-18': ('21:00', -3), '2021-06-21': ('21:00', -3), '2021-06-28': ('20:00', -4),
    '2021-07-03': ('22:00', -3), '2021-07-06': ('22:00', -3), '2021-07-10': ('21:00', -3),
    '2021-09-02': ('20:00', -4), '2021-09-09': ('20:30', -3), '2021-10-07': ('20:30', -3),
    '2021-10-10': ('20:30', -3), '2021-10-14': ('20:30', -3), '2021-11-12': ('20:30', -3),
    '2021-11-16': ('20:30', -3), '2022-01-27': ('21:15', -3), '2022-02-01': ('20:30', -3),
    '2022-03-25': ('20:30', -3), '2022-03-29': ('18:30', -5), '2022-06-01': ('19:45', 1),
    '2022-06-05': ('19:00', 2), '2022-09-23': ('21:00', -4), '2022-09-27': ('21:00', -4),
    '2022-11-16': ('19:30', 4), '2022-11-22': ('13:00', 3), '2022-11-26': ('22:00', 3),
    '2022-11-30': ('22:00', 3), '2022-12-03': ('22:00', 3), '2022-12-09': ('22:00', 3),
    '2022-12-13': ('22:00', 3), '2022-12-18': ('18:00', 3), '2023-03-23': ('20:30', -3),
    '2023-03-28': ('20:30', -3), '2023-06-15': ('20:00', 8), '2023-06-19': ('19:30', 7),
    '2023-09-07': ('21:00', -3), '2023-09-12': ('16:00', -4), '2023-10-12': ('21:00', -3),
    '2023-10-17': ('21:00', -5), '2023-11-16': ('21:00', -3), '2023-11-21': ('21:30', -3),
    '2024-03-22': ('20:00', -4), '2024-03-26': ('19:50', -7), '2024-06-09': ('17:00', -5),
    '2024-06-14': ('20:00', -4), '2024-06-20': ('20:00', -4), '2024-06-25': ('21:00', -4),
    '2024-06-29': ('20:00', -4), '2024-07-04': ('20:00', -5), '2024-07-09': ('20:00', -4),
    '2024-07-14': ('21:22', -4), '2024-09-05': ('21:00', -3), '2024-09-10': ('17:30', -5),
    '2024-10-10': ('17:00', -4), '2024-10-15': ('21:00', -3), '2024-11-14': ('20:30', -3),
    '2024-11-19': ('21:45', -3), '2025-03-21': ('20:30', -3), '2025-03-25': ('21:00', -3),
    '2025-06-05': ('21:00', -4), '2025-06-10': ('21:00', -3), '2025-09-04': ('20:30', -3),
    '2025-09-09': ('18:00', -5), '2025-10-10': ('20:00', -4), '2025-10-14': ('18:00', -4),
    '2025-11-14': ('17:00', 1),
}
con_alargue = {'2022-12-09', '2022-12-18', '2024-07-14'}  # empatados a los 90' y con tiempo extra

arg['fecha'] = arg['date'].dt.strftime('%Y-%m-%d')
print('Partidos sin horario:', len(set(arg['fecha']) - set(horarios)))
inicio_local = pd.to_datetime(arg['fecha'] + ' ' + arg['fecha'].map(lambda f: horarios[f][0]))
huso = arg['fecha'].map(lambda f: horarios[f][1])
arg['inicio'] = inicio_local - pd.to_timedelta(huso + 3, unit='h')
arg['hora_arg'] = arg['inicio'].dt.strftime('%H:%M')

# 3. Goles y condición desde Argentina
es_local = arg['home_team'] == 'Argentina'
arg['rival'] = np.where(es_local, arg['away_team'], arg['home_team'])
arg['goles_a_favor'] = np.where(es_local, arg['home_score'], arg['away_score'])
arg['goles_en_contra'] = np.where(es_local, arg['away_score'], arg['home_score'])
arg['condicion'] = np.select([arg['neutral'], es_local], ['Neutral', 'Home'], 'Away')

# 4. Penales y resultado final
pen = penales_raw[(penales_raw['home_team'] == 'Argentina') | (penales_raw['away_team'] == 'Argentina')]
ganador = pen.set_index(pen['date'].dt.strftime('%Y-%m-%d'))['winner']
arg['penales'] = arg['fecha'].isin(ganador.index)
arg['alargue'] = arg['fecha'].isin(con_alargue)
arg['resultado'] = np.select([arg['goles_a_favor'] > arg['goles_en_contra'], arg['goles_a_favor'] < arg['goles_en_contra']],
                             ['Victoria', 'Derrota'], 'Empate')
gano = arg['fecha'].map(ganador) == 'Argentina'
arg.loc[arg['penales'] & gano, 'resultado'] = 'Victoria'
arg.loc[arg['penales'] & ~gano, 'resultado'] = 'Derrota'

seleccion = (arg.rename(columns={'tournament': 'competencia'})
             [['fecha', 'hora_arg', 'inicio', 'rival', 'competencia', 'condicion', 'city', 'country',
               'goles_a_favor', 'goles_en_contra', 'resultado', 'alargue', 'penales']]
             .sort_values('inicio').reset_index(drop=True))
print(seleccion['resultado'].value_counts())
seleccion[seleccion['inicio'].dt.hour < 18][['fecha', 'hora_arg', 'rival', 'competencia']]

Partidos sin horario: 0
resultado
Victoria    54
Empate       8
Derrota      5
Name: count, dtype: int64


,fecha,hora_arg,rival,competencia
20,2022-06-01,15:45,Italy,CONMEBOL–UEFA Cup of Champions
21,2022-06-05,14:00,Estonia,Friendly
24,2022-11-16,12:30,United Arab Emirates,Friendly
25,2022-11-22,07:00,Saudi Arabia,FIFA World Cup
26,2022-11-26,16:00,Mexico,FIFA World Cup
27,2022-11-30,16:00,Poland,FIFA World Cup
28,2022-12-03,16:00,Australia,FIFA World Cup
29,2022-12-09,16:00,Netherlands,FIFA World Cup
30,2022-12-13,16:00,Croatia,FIFA World Cup
31,2022-12-18,12:00,France,FIFA World Cup


La tabla anterior muestra por qué no sirve asignar una hora fija: estos partidos empezaron de mañana o de tarde en Argentina (Mundial de Qatar, Finalissima, giras por Asia).

## 4. Cruce: siniestros en las 12 h posteriores a cada partido

**Método.** En la Ciudad hay muchos más siniestros a las 18 h que a las 4 h, y los sábados no se parecen a los martes. Por eso no alcanza con contar siniestros después de cada partido: hay que compararlos con **lo que se esperaría a esa hora, ese día de la semana y ese año, sin partido**. Para cada partido:

- **Observado:** siniestros en las 12 h siguientes al final del partido.
- **Esperado:** promedio de siniestros en ventanas de 12 h que arrancan a la misma hora, el mismo día de la semana y el mismo año, **sin ningún partido cerca** (ventanas de control).
- **Exceso = observado − esperado** y **ratio = observado / esperado** (1,10 = 10 % más de lo normal).

Primero se unen los 568 partidos en una sola tabla.

In [7]:
cols = ['equipo', 'inicio', 'fin', 'competencia', 'condicion', 'rival', 'goles_a_favor', 'goles_en_contra',
        'resultado', 'superclasico']
sel = seleccion.copy()
sel['equipo'] = 'Selección Argentina'
sel['superclasico'] = False
sel['fin'] = (sel['inicio'] + pd.Timedelta(minutes=115)
              + pd.to_timedelta(sel['alargue'] * 30 + sel['penales'] * 15, unit='min'))
partidos = pd.concat([river_boca[cols], sel[cols]], ignore_index=True).sort_values('inicio').reset_index(drop=True)
print('Partidos totales:', len(partidos))
partidos.groupby('equipo')['resultado'].value_counts().unstack()[ORDEN]

Partidos totales: 568


resultado,Derrota,Empate,Victoria
equipo,,,
Boca Juniors,65,64,120
River Plate,57,65,130
Selección Argentina,5,8,54


Se calcula, para cada hora del período, la suma de siniestros de las 12 h siguientes. Los 77 siniestros sin hora (0,16 %) no se pueden ubicar y quedan afuera.

In [8]:
horas = pd.date_range('2021-01-01 00:00', '2025-12-31 23:00', freq='h')
serie = (hechos.dropna(subset=['fecha_hora']).set_index('fecha_hora').resample('h').size()
         .reindex(horas, fill_value=0))
ventana12 = serie.rolling(12).sum().shift(-11)   # t, t+1, ..., t+11

base = pd.DataFrame({'siniestros_12h': ventana12, 'anio': horas.year, 'dia_semana': horas.dayofweek, 'hora': horas.hour},
                    index=horas)
print('Promedio de siniestros en 12 h:', round(ventana12.mean(), 2))

Promedio de siniestros en 12 h: 13.56


Una hora es **control** si su ventana no se superpone con la de ningún partido. El esperado de cada partido es el promedio de las ventanas de control con su mismo año, día de la semana y hora de inicio.

In [9]:
partidos['inicio_ventana'] = partidos['fin'].dt.floor('h')
contaminada = pd.Series(False, index=horas)
for w in partidos['inicio_ventana']:
    contaminada[(horas > w - pd.Timedelta(hours=12)) & (horas < w + pd.Timedelta(hours=12))] = True
base['control'] = ~contaminada & base['siniestros_12h'].notna()
print(f"Horas de control: {base['control'].sum():,} de {len(base):,} ({base['control'].mean():.0%})")

esperado = base[base['control']].groupby(['anio', 'dia_semana', 'hora'])['siniestros_12h'].mean().rename('esperado')
partidos['anio'] = partidos['inicio_ventana'].dt.year
partidos['dia_semana'] = partidos['inicio_ventana'].dt.dayofweek
partidos['hora'] = partidos['inicio_ventana'].dt.hour
partidos['observado'] = ventana12.reindex(partidos['inicio_ventana']).values
partidos = partidos.join(esperado, on=['anio', 'dia_semana', 'hora'])
partidos['exceso'] = partidos['observado'] - partidos['esperado']
partidos['ratio'] = partidos['observado'] / partidos['esperado']
partidos = partidos.dropna(subset=['observado', 'esperado']).reset_index(drop=True)
partidos[['equipo', 'inicio', 'rival', 'resultado', 'observado', 'esperado', 'ratio']].head()

Horas de control: 32,491 de 43,824 (74%)


,equipo,inicio,rival,resultado,observado,esperado,ratio
0,Boca Juniors,2021-02-14 19:20:00,Gimnasia–LP,Empate,2.0,6.047619,0.330709
1,River Plate,2021-02-14 21:30:00,Estudiantes–LP,Derrota,1.0,6.857143,0.145833
2,River Plate,2021-02-20 21:30:00,Rosario Central,Victoria,6.0,5.064516,1.184713
3,Boca Juniors,2021-02-21 21:30:00,Newell's,Victoria,4.0,6.857143,0.583333
4,River Plate,2021-02-28 17:10:00,Platense,Victoria,10.0,5.666667,1.764706


Para el gráfico hora por hora de 3.4 se calcula también el promedio de siniestros en cada una de las 12 horas posteriores al partido (observado y esperado), según el resultado.

In [10]:
esperado_h = base[base['control']].assign(s=serie).groupby(['anio', 'dia_semana', 'hora'])['s'].mean()
filas = []
for res in ORDEN:
    d = partidos[partidos['resultado'] == res]
    for k in range(12):
        t = d['inicio_ventana'] + pd.Timedelta(hours=k)
        filas.append({'resultado': res, 'horas_desde_fin': k, 'observado': serie.reindex(t).mean(),
                      'esperado': esperado_h.reindex(list(zip(t.dt.year, t.dt.dayofweek, t.dt.hour))).mean()})
perfil_horario = pd.DataFrame(filas)
perfil_horario.pivot(index='horas_desde_fin', columns='resultado', values='observado').round(2).T

horas_desde_fin,0,1,2,3,4,5,6,7,8,9,10,11
resultado,,,,,,,,,,,,
Derrota,0.89,0.80,0.73,0.58,0.52,0.41,0.31,0.36,0.61,0.86,1.03,0.98
Empate,0.96,0.89,0.80,0.58,0.49,0.34,0.28,0.36,0.45,0.60,0.91,1.14
Victoria,0.94,0.90,0.89,0.55,0.50,0.34,0.34,0.43,0.64,0.84,1.05,1.09


## 5. Resumen agrupado

Primer estadístico agrupado: siniestros observados, esperados, ratio y exceso según el **resultado** del partido.

In [11]:
resumen = (partidos.groupby('resultado')
           .agg(partidos=('ratio', 'size'), observado_medio=('observado', 'mean'), esperado_medio=('esperado', 'mean'),
                ratio_medio=('ratio', 'mean'), exceso_medio=('exceso', 'mean'))
           .reindex(ORDEN).round(2))
resumen

,partidos,observado_medio,esperado_medio,ratio_medio,exceso_medio
resultado,,,,,
Derrota,127,8.07,8.12,0.99,-0.05
Empate,137,7.79,7.84,1.01,-0.06
Victoria,304,8.53,8.18,1.04,0.35


Segundo estadístico agrupado: el mismo resumen por **equipo y resultado**.

In [12]:
(partidos.groupby(['equipo', 'resultado'])
 .agg(partidos=('ratio', 'size'), ratio_medio=('ratio', 'mean'), exceso_medio=('exceso', 'mean')).round(2))

partidos  ratio_medio  exceso_medio
equipo              resultado                                     
Boca Juniors        Derrota          65         0.99         -0.11
                    Empate           64         1.04          0.08
                    Victoria        120         1.07          0.48
River Plate         Derrota          57         0.98         -0.13
                    Empate           65         0.99         -0.17
                    Victoria        130         1.00          0.03
Selección Argentina Derrota           5         1.19          1.63
                    Empate            8         0.93         -0.22
                    Victoria         54         1.08          0.81

**Interpretación.** En las 12 h posteriores a una derrota se registraron en promedio **8,07 siniestros**, contra **8,12 esperados** para ese horario, día y año: un ratio de **0,99**, es decir, lo normal. Tras empates el ratio es 1,01 y tras victorias 1,04. Por equipo, las derrotas de Boca (0,99) y de River (0,98) tampoco muestran aumento. La Selección tiene un ratio alto en derrotas (1,19), pero con **solo 5 partidos**, demasiado pocos para sacar conclusiones. Llama la atención que los valores más altos entre los grupos grandes sean los de las **victorias** de la Selección (1,08) y de Boca (1,07), no los de las derrotas.

Las 5 derrotas de la Selección, caso por caso (con tan pocos casos no tiene sentido un test para la Selección sola):

In [13]:
(partidos[(partidos['equipo'] == 'Selección Argentina') & (partidos['resultado'] == 'Derrota')]
 [['inicio', 'rival', 'competencia', 'observado', 'esperado', 'ratio']].round({'esperado': 2, 'ratio': 2}))

,inicio,rival,competencia,observado,esperado,ratio
224,2022-11-22 07:00:00,Saudi Arabia,FIFA World Cup,22.0,22.22,0.99
340,2023-11-16 21:00:00,Uruguay,FIFA World Cup qualification,9.0,7.78,1.16
432,2024-09-10 19:30:00,Colombia,FIFA World Cup qualification,5.0,7.08,0.71
457,2024-11-14 20:30:00,Paraguay,FIFA World Cup qualification,11.0,8.67,1.27
540,2025-09-09 20:00:00,Ecuador,FIFA World Cup qualification,15.0,8.10,1.85


**Interpretación.** Los resultados son muy dispares: tras Ecuador (2025) hubo un 85 % más de siniestros de lo esperado y tras Paraguay (2024) un 27 % más, pero tras Colombia (2024) un 29 % menos, y tras Arabia Saudita en el Mundial, jugado un martes a las 7 de la mañana, los siniestros fueron los normales (22 contra 22,2). Con tan pocos casos y tanta dispersión no se puede afirmar que haya un efecto.

Como dato llamativo, el mayor exceso del período se dio tras una **victoria**: la **final del Mundial 2022**, con 19 siniestros contra 9,7 esperados, un día en que millones de personas salieron a festejar. Si el fútbol influye en los siniestros, parecería hacerlo más por la **movilización masiva** que por la frustración de perder.

Los partidos con mayor exceso de siniestros:

In [14]:
partidos.sort_values('ratio', ascending=False)[['equipo', 'inicio', 'rival', 'resultado', 'observado', 'esperado', 'ratio']].head(5).round({'esperado': 2, 'ratio': 2})

,equipo,inicio,rival,resultado,observado,esperado,ratio
465,River Plate,2024-12-08 19:15:00,Rosario Central,Victoria,21.0,7.92,2.65
54,Boca Juniors,2021-07-20 19:15:00,Atlético Mineiro,Derrota,12.0,4.92,2.44
137,Boca Juniors,2022-04-12 19:15:00,Always Ready,Victoria,17.0,7.25,2.34
98,River Plate,2021-10-31 20:15:00,Estudiantes–LP,Empate,14.0,6.19,2.26
109,Boca Juniors,2021-12-04 17:00:00,Arsenal,Empate,15.0,6.97,2.15


## 6. Pruebas de significación

1. **¿Las derrotas tienen más siniestros de lo esperado?** Wilcoxon (una muestra) sobre el exceso de las derrotas, contra 0.
2. **¿Las derrotas tienen más exceso que victorias y empates?** Mann-Whitney (no asume normalidad).
3. **Regresión de Poisson** (modelo para conteos): `observado ~ derrota + empate + equipo`, con el esperado como *offset*. El coeficiente de derrota exponenciado es el **cambio porcentual** de siniestros tras una derrota respecto de una victoria, ya controlado por hora, día y año. Se usa la versión *quasi-Poisson* para corregir la sobredispersión.

Nivel de significación: 5 %.

In [15]:
derrotas = partidos.loc[partidos['resultado'] == 'Derrota', 'exceso']
no_derrotas = partidos.loc[partidos['resultado'] != 'Derrota', 'exceso']
w = stats.wilcoxon(derrotas, alternative='greater')
mw = stats.mannwhitneyu(derrotas, no_derrotas, alternative='greater')
print(f'1) Wilcoxon exceso derrotas > 0:        p = {w.pvalue:.3f}  (exceso medio {derrotas.mean():+.2f} siniestros)')
print(f'2) Mann-Whitney derrotas > no derrotas:  p = {mw.pvalue:.3f}')

partidos['derrota'] = (partidos['resultado'] == 'Derrota').astype(int)
partidos['empate'] = (partidos['resultado'] == 'Empate').astype(int)
modelo = smf.glm('observado ~ derrota + empate + C(equipo)', data=partidos, family=sm.families.Poisson(),
                 offset=np.log(partidos['esperado'])).fit(scale='X2')
coef, ic = modelo.params['derrota'], modelo.conf_int().loc['derrota']
print(f'3) Poisson: derrota vs victoria = {np.exp(coef)-1:+.1%} siniestros '
      f'(IC 95%: {np.exp(ic[0])-1:+.1%} a {np.exp(ic[1])-1:+.1%}), p = {modelo.pvalues["derrota"]:.3f}')

1) Wilcoxon exceso derrotas > 0:        p = 0.721  (exceso medio -0.05 siniestros)
2) Mann-Whitney derrotas > no derrotas:  p = 0.786
3) Poisson: derrota vs victoria = -4.1% siniestros (IC 95%: -11.8% a +4.4%), p = 0.335


Salida completa del modelo:

In [16]:
modelo.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:              observado   No. Observations:                  568
Model:                            GLM   Df Residuals:                      563
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.3082
Method:                          IRLS   Log-Likelihood:                -1123.4
Date:                Wed, 04 Nov 2026   Deviance:                       747.51
Time:                        04:06:40   Pearson chi2:                     737.
No. Iterations:                     6   Pseudo R-squ. (CS):           0.007666
Covariance Type:            nonrobust                                         
====================================================================================================
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            0.0490      0.031      1.593      0.111      -0.011       0.109
C(equipo)[T.River Plate]            -0.0364      0.036     -1.008      0.313      -0.107       0.034
C(equipo)[T.Selección Argentina]     0.0373      0.052      0.710      0.478      -0.066       0.140
derrota                             -0.0414      0.043     -0.965      0.335      -0.126       0.043
empate                              -0.0406      0.042     -0.960      0.337      -0.124       0.042
====================================================================================================
"""

**Interpretación.** Ninguna de las tres pruebas encuentra un aumento significativo:

- El exceso tras las derrotas no es mayor que cero (Wilcoxon, p = 0,72).
- Las derrotas no tienen más exceso que victorias y empates (Mann-Whitney, p = 0,79).
- En la regresión de Poisson, una derrota se asocia con un **4,1 % menos** de siniestros que una victoria, pero el intervalo de confianza (−11,8 % a +4,4 %) incluye el cero (p = 0,34).

Con un 5 % de significación **no se rechaza H0**: no hay evidencia de que las derrotas aumenten los siniestros en las 12 h siguientes. El intervalo permite además descartar efectos grandes: un aumento mayor al ~4 % es poco compatible con los datos.

## 7. Robustez

Para comprobar que la conclusión no depende de una decisión particular del análisis, se repite la regresión cambiando una cosa por vez:

- **Sin superclásicos** (un equipo gana y el otro pierde en la misma ventana).
- **Solo River y Boca de local**, es decir, partidos jugados en la Ciudad.
- **Ventanas de 6 h y de 3 h**, por si el efecto fuera inmediato y se diluyera en 12 h.
- **Solo siniestros graves y mortales**, los más vinculados a conducción riesgosa.
- **Solo fines de semana.**

In [17]:
def analizar(hechos_df, subset, L=12):
    s = hechos_df.dropna(subset=['fecha_hora']).set_index('fecha_hora').resample('h').size().reindex(horas, fill_value=0)
    v = s.rolling(L).sum().shift(-(L - 1))
    cont = pd.Series(False, index=horas)
    for w in partidos['inicio_ventana']:
        cont[(horas > w - pd.Timedelta(hours=L)) & (horas < w + pd.Timedelta(hours=L))] = True
    b = pd.DataFrame({'v': v, 'a': horas.year, 'd': horas.dayofweek, 'h': horas.hour}, index=horas)
    esp = b[(~cont) & v.notna()].groupby(['a', 'd', 'h'])['v'].mean()
    q = subset.copy(); w = q['inicio_ventana']
    q['obs'] = v.reindex(w).values
    q['esp'] = esp.reindex(list(zip(w.dt.year, w.dt.dayofweek, w.dt.hour))).values
    q = q.dropna(subset=['obs', 'esp']).query('esp > 0')
    m = smf.glm('obs ~ derrota + empate + C(equipo)', data=q, family=sm.families.Poisson(),
                offset=np.log(q['esp'])).fit(scale='X2')
    ic = m.conf_int().loc['derrota']
    der = q[q['derrota'] == 1]
    return {'partidos': len(q), 'derrotas': len(der), 'ratio_derrotas': der['obs'].sum() / der['esp'].sum(),
            'efecto_derrota': np.exp(m.params['derrota']) - 1, 'ic_inf': np.exp(ic[0]) - 1, 'ic_sup': np.exp(ic[1]) - 1,
            'p_valor': m.pvalues['derrota']}

graves = hechos[hechos['gravedad_siniestro'].isin(['GRAVE', 'MORTAL'])]
variantes = {
    'Principal (12 h)': (hechos, partidos, 12),
    'Sin superclásicos': (hechos, partidos[~partidos['superclasico'].astype(bool)], 12),
    'Solo River y Boca de local (CABA)': (hechos, partidos[(partidos['condicion'] == 'Home') & (partidos['equipo'] != 'Selección Argentina')], 12),
    'Ventana de 6 h': (hechos, partidos, 6),
    'Ventana de 3 h': (hechos, partidos, 3),
    'Solo siniestros graves y mortales': (graves, partidos, 12),
    'Solo fines de semana': (hechos, partidos[partidos['inicio'].dt.dayofweek >= 5], 12),
}
robustez = pd.DataFrame({k: analizar(*a) for k, a in variantes.items()}).T
robustez['significativo'] = np.where(robustez['p_valor'] < 0.05, 'Sí', 'No')
fmt = robustez.copy()
for c in ['efecto_derrota', 'ic_inf', 'ic_sup']:
    fmt[c] = (fmt[c].astype(float) * 100).round(1).astype(str) + ' %'
fmt[['partidos', 'derrotas']] = fmt[['partidos', 'derrotas']].astype(int)
fmt['ratio_derrotas'] = fmt['ratio_derrotas'].astype(float).round(2)
fmt['p_valor'] = fmt['p_valor'].astype(float).round(2)
fmt

,partidos,derrotas,ratio_derrotas,efecto_derrota,ic_inf,ic_sup,p_valor,significativo
Principal (12 h),568,127,0.99,-4.1 %,-11.8 %,4.4 %,0.33,No
Sin superclásicos,544,117,0.99,-4.6 %,-12.6 %,4.1 %,0.29,No
Solo River y Boca de local (CABA),253,42,1.09,6.7 %,-6.4 %,21.6 %,0.33,No
Ventana de 6 h,568,127,0.96,-2.9 %,-13.6 %,9.0 %,0.62,No
Ventana de 3 h,568,127,0.92,-9.7 %,-21.9 %,4.3 %,0.17,No
Solo siniestros graves y mortales,568,127,0.87,-19.5 %,-38.6 %,5.4 %,0.12,No
Solo fines de semana,310,71,1.00,-5.7 %,-15.8 %,5.7 %,0.31,No


**Interpretación.** En las siete variantes el efecto de la derrota es **no significativo** y su intervalo de confianza incluye el cero. El único caso con ratio por encima de 1 es el de River y Boca de local (1,09), pero con 42 derrotas el intervalo es amplio (−6 % a +22 %) y no alcanza significación. Con ventanas más cortas o mirando solo siniestros graves, el efecto incluso va en sentido contrario (menos siniestros tras perder), tampoco significativo. **La conclusión no depende de las decisiones metodológicas.**

## 8. Exportación

Se guardan las bases limpias y los resultados del cruce. Las bases ya están en la carpeta `datos/` del repositorio; los dos archivos de resultados (`partidos_con_siniestros_12h.csv` y `perfil_horario_12h.csv`) son los que usa el notebook **3.4**.

In [18]:
hechos.to_csv('hechos_limpio_2021_2025.csv', index=False)
victimas.to_csv('victimas_limpio_2021_2025.csv', index=False)
siniestros_por_hora.to_csv('siniestros_por_hora_2021_2025.csv', index=False)
river_boca[river_boca['equipo'] == 'Boca Juniors'].to_csv('boca_2021_2025.csv', index=False)
river_boca[river_boca['equipo'] == 'River Plate'].to_csv('river_2021_2025.csv', index=False)
river_boca.to_csv('river_boca_2021_2025.csv', index=False)
seleccion.drop(columns=['inicio']).to_csv('partidos_seleccion_2021_2025.csv', index=False)
partidos.drop(columns=['derrota', 'empate']).to_csv('partidos_con_siniestros_12h.csv', index=False)
perfil_horario.to_csv('perfil_horario_12h.csv', index=False)
print('Archivos guardados.')

Archivos guardados.
